Para iniciar, el sistema requiere generar dos numeros primos de gran longitud. Como buscar divisores uno por uno tomaria una cantidad incalculable de tiempo, el algoritmo implementa el test de primalidad de Miller-Rabin. Este es un metodo probabilistico que determina si un numero enorme es primo ejecutando una serie de operaciones modulares a gran velocidad.

El proceso comienza generando cadenas de bits completamente aleatorias. El sistema toma estos bits y asegura que el ultimo de ellos sea un uno, lo que fuerza a que el numero resultante sea impar, ya que un numero par nunca podria ser primo a excepcion del dos. Una vez construido este candidato impar, se somete al test de Miller-Rabin. Si el candidato falla la prueba, el algoritmo lo descarta y genera una nueva cadena de bits. Este ciclo se repite ininterrumpidamente hasta que el test confirma que se ha encontrado un numero primo valido.

Al obtener los dos numeros primos gigantes, los cuales se identifican en la formula como p y q, el sistema procede a estructurar las llaves. Para lograr una llave de 1024 bits, el algoritmo fuerza a que tanto p como q tengan una longitud de 512 bits cada uno.

El primer calculo consiste en multiplicar p por q para obtener el modulo, identificado como n. Este numero define el limite maximo de tamaño para cualquier operacion matematica posterior y se hace publico, ya que integra ambas llaves.

A continuacion, se calcula la funcion totiente de Euler, conocida como phi. Esto se logra multiplicando p menos uno por q menos uno. El valor resultante de phi describe una propiedad geometrica y numerica esencial del modulo n y debe mantenerse en secreto absoluto por el sistema.

Posteriormente, el algoritmo establece el exponente publico e. Se selecciona el numero 65537, un estandar en la industria por su eficiencia en calculos binarios, y se verifica mediante una funcion de maximo comun divisor que no comparta factores con phi.

El paso final de esta fase es calcular el exponente privado d. Este valor es el inverso multiplicativo de e bajo el modulo phi. En terminos practicos, es el unico numero existente que, al ser multiplicado por el exponente publico e y luego dividido entre phi, deja un residuo exacto de uno.

Tras estos calculos, el sistema agrupa el exponente publico e junto con el modulo n para formar la llave publica. Por otro lado, agrupa el exponente privado d con el mismo modulo n para formar la llave privada.

Cuando un autor necesita firmar un archivo, el sistema no opera sobre el texto completo. En su lugar, el algoritmo somete el archivo a la funcion de hash SHA-256. Este proceso de codificacion de una sola via comprime toda la informacion del documento original, sin importar su tamaño, en un resumen unico de 256 bits que actua como una huella digital.

Dado que las llaves operan estrictamente mediante matematicas, la huella digital en formato de bytes es convertida por el sistema a un numero entero gigante. Como el modulo n calculado previamente tiene un tamaño de 1024 bits, el numero entero derivado del hash de 256 bits es significativamente menor, lo cual es un requisito indispensable para que la formula funcione sin perdida de informacion.

Para crear la firma digital, el algoritmo toma el numero entero del hash y lo eleva a la potencia del exponente privado d, calculando el residuo de la division de este resultado gigantesco entre el modulo n. La cifra resultante es la firma digital final, un numero que solo pudo ser generado por quien posee el exponente privado.

El flujo concluye cuando un receptor necesita comprobar que el documento es legitimo. El sistema del receptor toma el documento tal como llego y lo procesa nuevamente por el algoritmo SHA-256, convirtiendo el resultado en un numero entero.

En paralelo, el algoritmo toma la firma digital recibida y realiza la operacion matematica inversa. Eleva el numero de la firma a la potencia del exponente publico e y aplica el modulo n.

El paso definitivo ocurre cuando el sistema compara el numero entero del hash calculado a partir del documento recibido contra el numero extraido de la firma digital. Si ambas cifras son exactamente iguales, el algoritmo dictamina que el documento es autentico y su integridad esta intacta. Si un solo caracter del archivo original hubiese sido modificado, el primer hash arrojaria un numero totalmente distinto y la comparacion final fracasaria, alertando al sistema sobre la alteracion.

In [3]:
import hashlib
import random
import math

def es_primo(n, k=5):
    if n == 2 or n == 3:
        return True
    if n <= 1 or n % 2 == 0:
        return False
    s = n - 1
    r = 0
    while s % 2 == 0:
        s //= 2
        r += 1
    for _ in range(k):
        a = random.randrange(2, n - 1)
        x = pow(a, s, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def generar_primo_gigante(bits):
    while True:
        p = random.getrandbits(bits)
        p |= (1 << bits - 1) | 1
        if es_primo(p):
            return p

def generar_llaves_rsa(bits=1024):
    print(f"Generando primos 'p' y 'q' de {bits//2} bits (esto puede tomar un segundo)...")
    p = generar_primo_gigante(bits // 2)
    q = generar_primo_gigante(bits // 2)
    n = p * q
    phi = (p - 1) * (q - 1)
    e = 65537
    if math.gcd(e, phi) != 1:
        while math.gcd(e, phi) != 1:
            e += 2
    d = pow(e, -1, phi)
    return (e, n), (d, n)

def firmar_documento(documento, llave_privada):
    d, n = llave_privada
    hash_completo = hashlib.sha256(documento).digest()
    hash_int = int.from_bytes(hash_completo, byteorder='big')
    firma = pow(hash_int, d, n)
    return firma

def verificar_firma(documento, firma, llave_publica):
    e, n = llave_publica
    hash_completo = hashlib.sha256(documento).digest()
    hash_calculado_int = int.from_bytes(hash_completo, byteorder='big')
    hash_descifrado_int = pow(firma, e, n)
    return hash_calculado_int == hash_descifrado_int

print("--- 1. CREACION DE LLAVES ---")
llave_pub, llave_priv = generar_llaves_rsa(bits=1024)
print("Llaves generadas con exito.\n")

documento_original = b"Acuerdo de confidencialidad y pago por $50,000"

print("--- 2. PROCESO DE FIRMA ---")
firma_digital = firmar_documento(documento_original, llave_priv)
print(f"Firma digital generada (primeros 50 digitos): {str(firma_digital)[:50]}...\n")

print("--- 3. VERIFICACION ---")
if verificar_firma(documento_original, firma_digital, llave_pub):
    print("RESULTADO: La firma es valida. El documento es el original.\n")
else:
    print("RESULTADO: Firma invalida.\n")

print("--- 4. VERIFICACION ---")
documento_alterado = b"Acuerdo de confidencialidad y pago por $90,000"
if verificar_firma(documento_alterado, firma_digital, llave_pub):
    print("RESULTADO: La firma es valida.")
else:
    print("RESULTADO ESPERADO: La verificacion fallo. El documento fue alterado.")

--- 1. CREACION DE LLAVES ---
Generando primos 'p' y 'q' de 512 bits (esto puede tomar un segundo)...
Llaves generadas con exito.

--- 2. PROCESO DE FIRMA ---
Firma digital generada (primeros 50 digitos): 17118865986605710994164430217749561914125229541231...

--- 3. VERIFICACION ---
RESULTADO: La firma es valida. El documento es el original.

--- 4. VERIFICACION ---
RESULTADO ESPERADO: La verificacion fallo. El documento fue alterado.
